# Water Leak Detection — Portfolio Audit

Detects slow (Minimum Night Flow) and sudden (burst) leaks per customer meter, with:
- category-aware, per-customer-scaled thresholds
- gap-aware, time-true rolling windows
- an explicit night-trough applicability check for 24/7 accounts
- data completeness reporting
- auto-detection of the real header row (handles files with title/metadata rows above the actual columns)

Run the cells top to bottom. Point `base_folder` in the last audit cell at your actual data root.

## 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np

## 2. Header-row auto-detection helper

Handles files that have extra title/metadata rows above the real column headers.

In [2]:
def _detect_header_row(file_path, time_col, consumption_col, max_scan_rows=10):
    """
    Scans the first `max_scan_rows` rows of the file (with no header assumed) and
    returns the index of the first row that actually contains both `time_col` and
    `consumption_col` as literal cell values. This handles files that have extra
    title/metadata rows above the real column headers (a common export quirk),
    without needing a fixed header=N assumption that only works for some files.
    Falls back to row 0 if nothing matches, so behavior is unchanged for files
    that already have a clean header on the first row.
    """
    try:
        if file_path.endswith('.csv'):
            preview = pd.read_csv(file_path, header=None, nrows=max_scan_rows)
        else:
            preview = pd.read_excel(file_path, header=None, nrows=max_scan_rows)
    except Exception:
        return 0

    for i in range(len(preview)):
        row_vals = preview.iloc[i].astype(str).str.strip().tolist()
        if time_col in row_vals and consumption_col in row_vals:
            return i
    return 0

## 3. Core per-customer leak detection function

In [3]:
def analyze_leak_production_grade(file_path, folder_type, time_col='Hourly', consumption_col='Consumption m3'):
    """
    High-fidelity water leak detection engine.

    v3 changes vs prior version:
      1. Std floor for z-scores is computed per (day_of_week, hour) group instead of
         one global scalar -> night hours (naturally low variance) aren't drowned out
         by daytime variance when picking the floor.
      2. Night-leak drift threshold scales to each customer's OWN historical night
         floor (category gives a multiplier + absolute backstop, not one fixed m3
         value for every customer in that category).
      3. Data is reindexed onto a complete expected hourly grid before any rolling
         logic, so "7 days" / "N consecutive hours" are true wall-clock windows and
         not just "N rows", which previously could be silently wrong across data gaps.
      4. Data completeness is measured explicitly (recent window + night hours) and
         returned in the output, and an `MNF_Evaluated` flag distinguishes
         "confirmed normal" from "could not evaluate reliably due to missing data".
    """
    try:
        # --- 1. DATA INGESTION & ROBUST HYGIENE ---
        header_row = _detect_header_row(file_path, time_col, consumption_col)
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path, header=header_row)
        else:
            df = pd.read_excel(file_path, header=header_row)

        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).drop_duplicates(subset=[time_col]).reset_index(drop=True)

        # --- 2. REINDEX TO A COMPLETE HOURLY GRID (gap-safety for all rolling logic) ---
        full_index = pd.date_range(df[time_col].min(), df[time_col].max(), freq='h')
        df = df.set_index(time_col).reindex(full_index)
        df.index.name = time_col
        df = df.reset_index()
        # consumption_col is now NaN wherever a reading was missing -- this is intentional
        # and tracked, rather than silently treated as "normal" later on.

        df['hour'] = df[time_col].dt.hour
        df['day_of_week'] = df[time_col].dt.dayofweek

        # --- 3. DYNAMIC OPERATIONAL CONFIGURATIONS ---
        # night_drift_multiplier: recent floor must exceed historical floor by this
        # multiple before being flagged. night_drift_abs_floor: backstop so tiny
        # historical floors (near-zero) still require a minimum real volume increase.
        if any(kw in folder_type for kw in ["Residential", "Villa", "Flat"]):
            night_hours = [1, 2, 3, 4]
            night_drift_multiplier = 1.6
            night_drift_abs_floor = 0.020
            z_threshold = 4.0
            consecutive_hours = 4
            burst_vol_threshold = 0.150
            std_floor_pct = 0.15

        elif "Government" in folder_type:
            night_hours = [0, 1, 5, 6]
            night_drift_multiplier = 1.5
            night_drift_abs_floor = 0.100
            z_threshold = 4.5
            consecutive_hours = 6
            burst_vol_threshold = 0.800
            std_floor_pct = 0.15

        elif "Industrial" in folder_type:
            night_hours = [2, 3, 4]
            night_drift_multiplier = 1.4
            night_drift_abs_floor = 0.180
            z_threshold = 5.0
            consecutive_hours = 6
            burst_vol_threshold = 1.500
            std_floor_pct = 0.15

        else:  # Commercial / Hotels
            night_hours = [2, 3, 4]
            night_drift_multiplier = 1.5
            night_drift_abs_floor = 0.090
            z_threshold = 4.0
            consecutive_hours = 4
            burst_vol_threshold = 0.500
            std_floor_pct = 0.15

        # --- 4. TIMELINE SEGREGATION & SANITY CHECKS ---
        max_date = df[time_col].max()
        four_weeks_ago = max_date - pd.Timedelta(weeks=4)
        eight_weeks_prior = four_weeks_ago - pd.Timedelta(weeks=8)

        min_date = df[time_col].min()
        if min_date >= four_weeks_ago:
            return {
                "Status": "SKIPPED",
                "Leak_Suspected": "NO",
                "Details": "Insufficient historical data footprint to map normal behavior profiles safely.",
                "MNF_Applicable": None,
                "Night_Trough_Ratio": None,
                "MNF_Evaluated": False,
                "Data_Completeness_Recent": None,
                "Data_Completeness_Night": None,
            }

        df['period'] = 'Ignore'
        df.loc[(df[time_col] >= eight_weeks_prior) & (df[time_col] < four_weeks_ago), 'period'] = 'Historical_Baseline'
        df.loc[df[time_col] >= four_weeks_ago, 'period'] = 'Recent_Evaluation'

        # --- 5. DATA COMPLETENESS (now meaningful, since the grid is complete) ---
        recent_mask = df['period'] == 'Recent_Evaluation'
        recent_total_rows = int(recent_mask.sum())
        recent_present_rows = int(df.loc[recent_mask, consumption_col].notna().sum())
        data_completeness_recent = round(recent_present_rows / recent_total_rows, 3) if recent_total_rows else 0.0

        recent_night_mask = recent_mask & df['hour'].isin(night_hours)
        recent_night_total = int(recent_night_mask.sum())
        recent_night_present = int(df.loc[recent_night_mask, consumption_col].notna().sum())
        data_completeness_night = round(recent_night_present / recent_night_total, 3) if recent_night_total else 0.0

        # --- 6. BEHAVIOR BASELINE COMPOSITION ---
        baseline_data = df[df['period'] == 'Historical_Baseline']

        baseline_profile = baseline_data.groupby(['day_of_week', 'hour'])[consumption_col].median().reset_index(name='baseline_median')
        baseline_std = baseline_data.groupby(['day_of_week', 'hour'])[consumption_col].std().reset_index(name='baseline_std')

        df = pd.merge(df, baseline_profile, on=['day_of_week', 'hour'], how='left')
        df = pd.merge(df, baseline_std, on=['day_of_week', 'hour'], how='left')

        # Per-(day_of_week, hour) std floor, scaled off that cell's own median rather
        # than a single global number. Night cells (naturally near-zero, low variance)
        # get a small floor; daytime cells get a floor proportional to their own scale.
        df['group_std_floor'] = df['baseline_median'].clip(lower=0) * std_floor_pct
        df['group_std_floor'] = df['group_std_floor'].clip(lower=0.010)

        # --- 7. STATISTICAL DEVIATION MEASUREMENT ---
        df['abs_deviation'] = df[consumption_col] - df['baseline_median']
        df['z_score'] = df['abs_deviation'] / (df['baseline_std'].fillna(0) + df['group_std_floor'])
        # Missing consumption readings must NOT collapse to "0 deviation" -- that would
        # make a dead meter look identical to confirmed-normal usage. Keep them NaN and
        # exclude from anomaly logic explicitly (handled by is_anomaly's NaN-safe compare).
        df.loc[df[consumption_col].isna(), ['abs_deviation', 'z_score']] = np.nan

        # --- 8. CORE LEAK EVALUATION ---
        leak_detected = False
        leak_reasons = []
        info_notes = []

        # --- VERIFICATION PATH A: SLOW & CONSTANT LEAKS (time-true, gap-aware) ---
        hist_night = baseline_data[baseline_data['hour'].isin(night_hours)]
        historical_min_flow = hist_night[consumption_col].quantile(0.05) if not hist_night.empty else 0.0

        # --- NIGHT-TROUGH APPLICABILITY CHECK ---
        # Minimum Night Flow only makes sense for accounts that actually go quiet at
        # night. A 24/7 site (e.g. a 3-shift industrial plant, a hospital, a hotel with
        # heavy night HVAC/laundry) never drops to a true trough, so a small leak is a
        # tiny fraction of a large, noisy continuous flow -- MNF has no real signal to
        # find there, and treating a "no drift detected" result as "no leak" would be
        # misleading. Estimated per customer (their own night floor vs. their own
        # typical all-hours usage) rather than assumed from category alone, since some
        # "Industrial" accounts are single-shift and some "Commercial" accounts run 24/7.
        NIGHT_TROUGH_RATIO_THRESHOLD = 0.55
        baseline_all_median = baseline_data[consumption_col].median()
        if baseline_all_median and baseline_all_median > 0:
            night_trough_ratio = historical_min_flow / baseline_all_median
        else:
            night_trough_ratio = np.nan

        mnf_applicable = True
        if pd.notna(night_trough_ratio) and night_trough_ratio > NIGHT_TROUGH_RATIO_THRESHOLD:
            mnf_applicable = False
            info_notes.append(
                f"Minimum Night Flow not applicable: night floor is {night_trough_ratio:.0%} of typical "
                f"usage (no real night trough, likely continuous/24-7 operation). Slow-leak detection for "
                f"this account relies on burst/statistical checks only -- treat a 'NO' leak result with "
                f"reduced confidence for slow, low-volume leaks."
            )

        recent_night = df[recent_night_mask].copy()
        highest_observed_floor = np.nan
        mnf_evaluated = False

        if mnf_applicable and not recent_night.empty:
            # Daily minimum night flow, but only trust a day if most expected night
            # readings for that day are actually present.
            recent_night['date'] = recent_night[time_col].dt.floor('D')
            day_group = recent_night.groupby('date')[consumption_col]
            daily_min = day_group.min()
            daily_coverage = day_group.apply(lambda s: s.notna().mean())
            daily_min = daily_min[daily_coverage >= 0.75]  # require >=75% of night hours present that day

            if not daily_min.empty:
                daily_min = daily_min.sort_index()
                # True 7-calendar-day rolling window (not row count) over daily minimums,
                # requiring at least 5 of 7 days present in a window to trust it.
                rolling_week_min = daily_min.rolling('7D', min_periods=5).min()
                if rolling_week_min.notna().any():
                    highest_observed_floor = rolling_week_min.max()
                    mnf_evaluated = True

        if mnf_evaluated:
            night_drift_threshold = max(night_drift_abs_floor, historical_min_flow * (night_drift_multiplier - 1))
            if highest_observed_floor > (historical_min_flow + night_drift_threshold) and highest_observed_floor > 0.02:
                leak_detected = True
                leak_reasons.append(
                    f"Slow Constant Leak: Night minimum floor structurally rose from "
                    f"{historical_min_flow:.3f} to {highest_observed_floor:.3f} m3 "
                    f"(threshold drift: {night_drift_threshold:.3f} m3)."
                )

        # --- VERIFICATION PATH B: SUDDEN MECHANICAL BURSTS (time-true consecutiveness) ---
        df['is_anomaly'] = (
            (df['period'] == 'Recent_Evaluation') &
            (df['z_score'] > z_threshold) &
            (df['abs_deviation'] > burst_vol_threshold)
        ).fillna(False)

        anomaly_series = df.set_index(time_col)['is_anomaly'].astype(int)
        # Time-based rolling window with min_periods == consecutive_hours: since the
        # index is now a complete hourly grid, this can only reach consecutive_hours
        # if that many TRUE hourly points genuinely fall inside the trailing window --
        # a gap in the middle breaks the streak instead of being silently skipped over.
        window_str = f"{consecutive_hours}h"
        consecutive_sum = anomaly_series.rolling(window_str, min_periods=consecutive_hours).sum()

        if (consecutive_sum >= consecutive_hours).any():
            leak_detected = True
            hit_times = consecutive_sum[consecutive_sum >= consecutive_hours].index
            window_rows = df[df[time_col].isin(hit_times)]
            max_z = window_rows['z_score'].max()
            peak_volume = window_rows[consumption_col].max()
            leak_reasons.append(
                f"Sudden Pipe Burst: Sustained, uncharacteristic high volume event for "
                f"{consecutive_hours}+ consecutive hours (Peak Vol: {peak_volume:.3f} m3, Peak Z: {max_z:.1f})."
            )

        # --- 9. UNIFIED OPERATIONAL OUTPUT ---
        detail_parts = leak_reasons if leak_reasons else ["Normal consumer operational patterns detected."]
        detail_parts = detail_parts + info_notes  # advisory notes shown regardless of verdict

        return {
            "Status": "SUCCESS",
            "Leak_Suspected": "YES" if leak_detected else "NO",
            "Details": " | ".join(detail_parts),
            "Historical_Night_Floor_m3": round(historical_min_flow, 4),
            "Recent_Night_Floor_m3": round(highest_observed_floor, 4) if pd.notna(highest_observed_floor) else None,
            "MNF_Applicable": mnf_applicable,
            "Night_Trough_Ratio": round(night_trough_ratio, 3) if pd.notna(night_trough_ratio) else None,
            "MNF_Evaluated": mnf_evaluated,
            "Data_Completeness_Recent": data_completeness_recent,
            "Data_Completeness_Night": data_completeness_night,
        }

    except Exception as e:
        return {
            "Status": "ERROR",
            "Leak_Suspected": "UNKNOWN",
            "Details": f"Execution Engine Failure: {str(e)}",
            "Historical_Night_Floor_m3": 0,
            "Recent_Night_Floor_m3": None,
            "MNF_Applicable": None,
            "Night_Trough_Ratio": None,
            "MNF_Evaluated": False,
            "Data_Completeness_Recent": None,
            "Data_Completeness_Night": None,
        }

## 4. Portfolio crawler

Walks a folder tree of customer files, runs the detector on each, and collects results.

In [4]:
def run_portfolio_leak_audit(base_folder="customers", time_col='Hourly', consumption_col='Consumption m3'):
    """
    Crawls through `base_folder`, finds every customer data file inside category
    subfolders (e.g. 'Villa (Residential)', 'Commercial', 'Government', ...),
    and runs analyze_leak_production_grade on each one.

    Each subfolder name is passed straight through as `folder_type`, so the
    category keyword matching inside analyze_leak_production_grade (looking for
    "Residential", "Villa", "Government", "Industrial", else Commercial) is driven
    by however you've actually named your folders -- rename folders, not code, to
    retarget a customer set to a different category profile.
    """
    results_list = []

    if not os.path.exists(base_folder):
        print(f"Directory path error: base folder '{base_folder}' does not exist.")
        return results_list

    print(f"Scanning '{base_folder}' for customer data files...\n")

    file_count = 0
    for root, dirs, files in os.walk(base_folder):
        for file in files:
            # Excel and CSV both supported by analyze_leak_production_grade
            is_excel = file.endswith(('.xlsx', '.xls')) and not file.startswith('~$')
            is_csv = file.endswith('.csv')
            if not (is_excel or is_csv):
                continue

            full_file_path = os.path.join(root, file)
            folder_category = os.path.basename(root)
            file_count += 1

            print(f"  [{file_count}] {full_file_path}  (profile: {folder_category})")

            # analyze_leak_production_grade already wraps its own logic in a
            # try/except and returns a Status: ERROR dict on failure, so one bad
            # file cannot crash the whole portfolio run. This outer try/except is
            # a second safety net only for something going wrong outside that
            # function (e.g. an unexpected exception raised while calling it).
            try:
                audit_summary = analyze_leak_production_grade(
                    file_path=full_file_path,
                    folder_type=folder_category,
                    time_col=time_col,
                    consumption_col=consumption_col,
                )
            except Exception as e:
                audit_summary = {
                    "Status": "ERROR",
                    "Leak_Suspected": "UNKNOWN",
                    "Details": f"Unhandled crawler-level failure: {str(e)}",
                    "Historical_Night_Floor_m3": 0,
                    "Recent_Night_Floor_m3": None,
                    "MNF_Applicable": None,
                    "Night_Trough_Ratio": None,
                    "MNF_Evaluated": False,
                    "Data_Completeness_Recent": None,
                    "Data_Completeness_Night": None,
                }

            audit_summary["Filename"] = file
            audit_summary["Full_Path"] = full_file_path
            audit_summary["Profile_Folder"] = folder_category
            results_list.append(audit_summary)

            if audit_summary.get("Status") == "ERROR":
                print(f"        -> ERROR: {audit_summary.get('Details')}")
            elif audit_summary.get("Status") == "SKIPPED":
                print(f"        -> SKIPPED: {audit_summary.get('Details')}")
            else:
                print(f"        -> {audit_summary.get('Leak_Suspected')}")

    if file_count == 0:
        print(f"No .xlsx / .xls / .csv files found anywhere under '{base_folder}'.")

    return results_list

## 5. Run the audit

Set `base_folder` to your actual data root (e.g. a folder containing subfolders like
`Villa (Residential)/`, `Commercial/`, `Government/`, `Industrial/`, each holding that
category's `.xlsx` / `.xls` / `.csv` meter files).

In [ ]:
all_audit_results = run_portfolio_leak_audit(base_folder="customers")

# 2. Turn results into a structured DataFrame if anything was found/processed.
if all_audit_results:
    summary_df = pd.DataFrame(all_audit_results)

    # Reorder columns so identifying info leads, then verdict, then diagnostics.
    preferred_order = [
        "Filename", "Profile_Folder", "Status", "Leak_Suspected", "Details",
        "Historical_Night_Floor_m3", "Recent_Night_Floor_m3",
        "MNF_Applicable", "Night_Trough_Ratio", "MNF_Evaluated",
        "Data_Completeness_Recent", "Data_Completeness_Night", "Full_Path",
    ]
    existing_cols = [c for c in preferred_order if c in summary_df.columns]
    remaining_cols = [c for c in summary_df.columns if c not in existing_cols]
    summary_df = summary_df[existing_cols + remaining_cols]

    # 3. Export to a master tracking CSV.
    output_file = "portfolio_leakage_audit_summary.csv"
    summary_df.to_csv(output_file, index=False)

    n_leaks = (summary_df["Leak_Suspected"] == "YES").sum()
    n_errors = (summary_df["Status"] == "ERROR").sum()
    n_skipped = (summary_df["Status"] == "SKIPPED").sum()
    n_mnf_na = (summary_df["MNF_Applicable"] == False).sum() if "MNF_Applicable" in summary_df else 0

    print(f"\nSUCCESS: audited {len(summary_df)} files -> exported to '{output_file}'")
    print(f"  Leaks suspected : {n_leaks}")
    print(f"  Errors          : {n_errors}")
    print(f"  Skipped (short) : {n_skipped}")
    print(f"  MNF not applicable (24/7-style accounts): {n_mnf_na}")
    print()
    print(summary_df.head())
else:
    print("Pipeline finished: no data records found or processed inside target directory tree.")

## 6. Inspect results in the notebook

`summary_df` is available after running the cell above — use it directly for further
filtering/plotting instead of only relying on the exported CSV.

In [ ]:
summary_df.sort_values('Leak_Suspected', ascending=False)